# 01. Quantum basics
States, inner products, Pauli and Hadamard gates, a first circuit, density matrices, partial trace and von Neumann entropy, using Qiskit.

Install once: `pip install qiskit qiskit-aer sympy matplotlib pylatexenc`

In [1]:
import numpy as np
from IPython.display import display, Math
from qiskit.quantum_info import Statevector, Operator, Pauli, DensityMatrix, partial_trace, entropy
from qiskit.visualization import array_to_latex
from qiskit.circuit.library import HGate

## A single-qubit state and basis vectors

In [2]:
u = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])   # the state |+>
display(u.draw("latex"))
ket0 = np.array([[1],[0]]); ket1 = np.array([[0],[1]])
display(array_to_latex(ket0)); display(array_to_latex(ket1))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## Inner product
$\langle 0|1\rangle = 0$: the basis states are orthogonal. `conj()` turns a ket into a bra.

In [3]:
v1 = Statevector([1,0]); v2 = Statevector([0,1])
print("Inner product:", np.vdot(v1.data, v2.data))

Inner product: 0j


## Pauli and Hadamard gates

In [4]:
X, Y, Z = Operator(Pauli('X')), Operator(Pauli('Y')), Operator(Pauli('Z'))
display(array_to_latex(X, prefix=r'\sigma_x = '))
display(array_to_latex(Y, prefix=r'\sigma_y = '))
display(array_to_latex(Z, prefix=r'\sigma_z = '))
H = Operator(HGate())
display(array_to_latex(H.data, prefix='H = '))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## A first circuit
A Hadamard puts the qubit in an equal superposition, so measuring gives 0 or 1 with probability about 1/2 each.

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
qc = QuantumCircuit(1,1)
qc.h(0); qc.measure(0,0)
print(AerSimulator().run(qc, shots=1000).result().get_counts())

{'1': 490, '0': 510}


## Tensor products
$|1\rangle \otimes |0\rangle$. Qiskit orders qubits little-endian, so `q0.expand(q1)` gives $|q_1 q_0\rangle$.

In [6]:
q0 = Statevector([1,0]); q1 = Statevector([0,1])
print(q0.expand(q1))

Statevector([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
            dims=(2, 2))


## Density matrices and partial trace
The two-qubit state $|{+}{+}\rangle$ is a product state, so tracing out qubit 1 leaves the pure state $\rho_+ = |+\rangle\langle +|$.

In [7]:
rho_plus = DensityMatrix(np.full((2,2), 0.5))
display(array_to_latex(rho_plus.data, prefix=r"\rho_+ = "))
rho_system = DensityMatrix(np.full((4,4), 0.25))
rho_reduced = partial_trace(rho_system, [1])
display(array_to_latex(rho_reduced.data, prefix=r"\rho_0 = "))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## Von Neumann entropy
$S(\rho) = -\mathrm{Tr}(\rho \log \rho)$, the quantum counterpart of Shannon entropy. A pure state has entropy 0. One half of a Bell pair has the maximum entropy of 1 bit.

In [8]:
from qiskit import QuantumCircuit
bell_circuit = QuantumCircuit(2)
bell_circuit.h(0); bell_circuit.cx(0, 1)
rho_bell = DensityMatrix(Statevector(bell_circuit))
print("Pure state entropy   :", round(float(entropy(rho_plus, base=2)), 6))
print("Bell pair, one qubit :", round(float(entropy(partial_trace(rho_bell, [1]), base=2)), 6))

Pure state entropy   : 0.0
Bell pair, one qubit : 1.0
